# 04. Usage Feature Engineering

이 노트북의 목적은 `02_preprocessing_policy.ipynb`에서 확정한 고객별 3주 관측창 데이터를 사용해, 구독 이벤트 단위의 시청 행동 파생변수를 생성하는 것이다.

이 단계는 영화 장르나 콘텐츠 메타데이터를 사용하지 않는다. 콘텐츠 메타데이터 기반 피처는 `05_content_feature_engineering.ipynb`에서 별도로 만든다.

핵심 원칙은 다음과 같다.

1. 분석 단위는 `membership_row_id`이다.
2. 시청 행동은 고객별 `reg_date` 기준 day 0~20 관측창 안의 기록만 사용한다.
3. 4주차 day 21~27은 리텐션 대응기간이므로 피처로 사용하지 않는다.
4. 시청이력이 없는 고객은 제거하지 않고, `no_watch_obs_flag`로 보존한다.
5. 단순 시청량뿐 아니라 초반 루틴화, 후반 몰아보기, 시청 공백, 주차별 변화량을 함께 만든다.

주요 출력 파일은 다음과 같다.

- `_data/02_interim/user_usage_features.csv`
- `_data/03_processed/modeling_feature_table_usage.csv`
- `_data/02_interim/usage_feature_summary.json`
- `reports/tables/04_usage_feature_*.csv`

## 4-1. 라이브러리 로딩

In [6]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

## 4-2. 경로 설정

이 노트북은 저장소 루트의 `_data`를 입력 저장소로만 사용한다.  
02번 노트북의 산출물은 `park.ingyeom/reports/data/02_preprocessing_policy`에서 읽고, 04번 산출물은 `park.ingyeom/reports/data/04_usage_feature_engineering` 및 `park.ingyeom/reports/tables/04_usage_feature_engineering`에 저장한다.


In [7]:
def find_project_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else Path(start).resolve()
    candidates = [start, *start.parents]

    for candidate in candidates:
        if (candidate / ".git").exists() and (candidate / "_data").exists():
            return candidate

    for candidate in candidates:
        if candidate.name == "park.ingyeom" and (candidate.parent / "_data").exists():
            return candidate.parent

    raise FileNotFoundError("저장소 루트를 찾지 못했습니다.")

PROJECT_ROOT = find_project_root()
DATA_ROOT = PROJECT_ROOT / "_data"
RAW_DIR = DATA_ROOT / "01_raw"
INTERIM_DIR = DATA_ROOT / "02_interim"

WORK_ROOT = PROJECT_ROOT / "park.ingyeom"
REPORTS_DIR = WORK_ROOT / "reports"

NOTEBOOK_ID = "04_usage_feature_engineering"

INPUT_DATA_DIR = REPORTS_DIR / "data" / "02_preprocessing_policy"
OUTPUT_DATA_DIR = REPORTS_DIR / "data" / NOTEBOOK_ID
OUTPUT_TABLE_DIR = REPORTS_DIR / "tables" / NOTEBOOK_ID

OUTPUT_DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_TABLE_DIR.mkdir(parents=True, exist_ok=True)

# Re-run safety: remove previous 04 outputs only inside the dedicated 04 folders.
previous_report_files = sorted(OUTPUT_TABLE_DIR.glob("04_*.csv"))
for path in previous_report_files:
    path.unlink()

previous_output_files = [
    OUTPUT_DATA_DIR / "usage_features.csv",
    OUTPUT_DATA_DIR / "modeling_feature_table_usage.csv",
    OUTPUT_DATA_DIR / "usage_feature_summary.json",
    OUTPUT_DATA_DIR / "user_usage_features.csv",  # legacy name, if any
]
removed_output_files = []
for path in previous_output_files:
    if path.exists():
        path.unlink()
        removed_output_files.append(path.name)

TABLES_DIR = OUTPUT_TABLE_DIR  # backward-compatible alias for existing table-saving cells.

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_ROOT:", DATA_ROOT)
print("INPUT_DATA_DIR:", INPUT_DATA_DIR)
print("OUTPUT_DATA_DIR:", OUTPUT_DATA_DIR)
print("OUTPUT_TABLE_DIR:", OUTPUT_TABLE_DIR)
print("Removed previous 04 report files:", len(previous_report_files))
print([p.name for p in previous_report_files])
print("Removed previous 04 output files:", len(removed_output_files))
print(removed_output_files)


PROJECT_ROOT: c:\Code\ott-churn-prediction
DATA_ROOT: c:\Code\ott-churn-prediction\_data
INPUT_DATA_DIR: c:\Code\ott-churn-prediction\park.ingyeom\reports\data\02_preprocessing_policy
OUTPUT_DATA_DIR: c:\Code\ott-churn-prediction\park.ingyeom\reports\data\04_usage_feature_engineering
OUTPUT_TABLE_DIR: c:\Code\ott-churn-prediction\park.ingyeom\reports\tables\04_usage_feature_engineering
Removed previous 04 report files: 2
['04_usage_feature_input_file_summary.csv', '04_usage_feature_observation_input_check.csv']
Removed previous 04 output files: 0
[]


## 4-3. 02번 산출물 로딩

이 노트북의 입력은 원본 CSV가 아니라 02번 전처리 노트북의 산출물이다.

In [8]:
INPUT_FILES = {
    "membership_preprocessed": INPUT_DATA_DIR / "membership_preprocessed.csv",
    "membership_with_usernum": INPUT_DATA_DIR / "membership_with_usernum.csv",
    "obs_view": INPUT_DATA_DIR / "view_history_observation_window.csv",
}

missing = [str(p) for p in INPUT_FILES.values() if not p.exists()]
if missing:
    raise FileNotFoundError(
        "02_preprocessing_policy.ipynb 산출물이 없습니다. 먼저 02번 노트북을 실행하세요. Missing: " + str(missing)
    )

membership = pd.read_csv(INPUT_FILES["membership_preprocessed"])
membership_user = pd.read_csv(INPUT_FILES["membership_with_usernum"])
obs_view = pd.read_csv(INPUT_FILES["obs_view"])

for col in ["reg_date", "end_date"]:
    if col in membership.columns:
        membership[col] = pd.to_datetime(membership[col], errors="coerce")
    if col in membership_user.columns:
        membership_user[col] = pd.to_datetime(membership_user[col], errors="coerce")

if "watch_day" in obs_view.columns:
    obs_view["watch_day"] = pd.to_datetime(obs_view["watch_day"], errors="coerce")

if "watch_time(min)" in obs_view.columns and "watch_time" not in obs_view.columns:
    obs_view = obs_view.rename(columns={"watch_time(min)": "watch_time"})

file_summary = pd.DataFrame([
    {"name": "membership_preprocessed", "path": str(INPUT_FILES["membership_preprocessed"]), "rows": len(membership), "cols": membership.shape[1]},
    {"name": "membership_with_usernum", "path": str(INPUT_FILES["membership_with_usernum"]), "rows": len(membership_user), "cols": membership_user.shape[1]},
    {"name": "view_history_observation_window", "path": str(INPUT_FILES["obs_view"]), "rows": len(obs_view), "cols": obs_view.shape[1]},
])

file_summary.to_csv(TABLES_DIR / "04_usage_feature_input_file_summary.csv", index=False, encoding="utf-8-sig")
display(file_summary)


,name,path,rows,cols
0,membership_preprocessed,c:\Code\ott-churn-prediction\park.ingyeom\repo...,14922,26
1,membership_with_usernum,c:\Code\ott-churn-prediction\park.ingyeom\repo...,14955,27
2,view_history_observation_window,c:\Code\ott-churn-prediction\park.ingyeom\repo...,85066,14


## 4-4. 입력 데이터 검산

04번은 시청 행동 피처를 만드는 단계이므로, 02번에서 정의한 관측창이 유지되어 있는지 먼저 확인한다.

In [9]:
required_membership_cols = {
    "membership_row_id",
    "USER_KEY",
    "is_repurchase",
    "is_100won",
    "max_screen",
}

required_obs_cols = {
    "membership_row_id",
    "USER_NUM",
    "MOVIE_NUM",
    "watch_day",
    "watch_time",
    "watch_rel_day",
    "obs_week",
}

missing_membership_cols = sorted(required_membership_cols - set(membership.columns))
missing_obs_cols = sorted(required_obs_cols - set(obs_view.columns))

if missing_membership_cols:
    raise ValueError(f"membership_preprocessed 필수 컬럼 누락: {missing_membership_cols}")
if missing_obs_cols:
    raise ValueError(f"view_history_observation_window 필수 컬럼 누락: {missing_obs_cols}")

watched_membership_ids = set(obs_view["membership_row_id"].dropna().unique())

membership["has_watch_obs"] = (
    membership["membership_row_id"].isin(watched_membership_ids).astype(int)
)
membership["no_watch_obs_flag"] = 1 - membership["has_watch_obs"]

obs_check = pd.DataFrame([
    {"item": "membership_rows", "value": len(membership)},
    {"item": "unique_membership_row_id_membership", "value": membership["membership_row_id"].nunique()},
    {"item": "obs_view_rows", "value": len(obs_view)},
    {"item": "unique_membership_row_id_obs_view", "value": obs_view["membership_row_id"].nunique()},
    {"item": "membership_has_watch_obs_sum", "value": int(membership["has_watch_obs"].sum())},
    {"item": "membership_no_watch_obs_sum", "value": int(membership["no_watch_obs_flag"].sum())},
    {"item": "obs_min_watch_rel_day", "value": obs_view["watch_rel_day"].min()},
    {"item": "obs_max_watch_rel_day", "value": obs_view["watch_rel_day"].max()},
    {"item": "obs_unique_movies", "value": obs_view["MOVIE_NUM"].nunique()},
    {"item": "obs_watch_time_sum", "value": obs_view["watch_time"].sum()},
])

obs_check.to_csv(
    TABLES_DIR / "04_usage_feature_observation_input_check.csv",
    index=False,
    encoding="utf-8-sig",
)

display(obs_check)

,item,value
0,membership_rows,14922
1,unique_membership_row_id_membership,14922
2,obs_view_rows,85066
3,unique_membership_row_id_obs_view,12302
4,membership_has_watch_obs_sum,12302
5,membership_no_watch_obs_sum,2620
6,obs_min_watch_rel_day,0
7,obs_max_watch_rel_day,20
8,obs_unique_movies,4765
9,obs_watch_time_sum,3800349


## 4-5. 기본 시청량 피처 생성

기본 시청량 피처는 고객별 총 시청시간, 세션 수, 고유 콘텐츠 수, 고유 시청일 수를 포함한다. 단순 시청량은 이전 분석에서 강한 신호가 아니었지만, 모델링과 검정의 기본 피처로 유지한다.

In [10]:
def build_basic_usage_features(obs: pd.DataFrame) -> pd.DataFrame:
    basic = obs.groupby('membership_row_id').agg(
        total_watch_time=('watch_time', 'sum'),
        total_sessions=('watch_time', 'count'),
        unique_contents=('MOVIE_NUM', 'nunique'),
        unique_days=('watch_rel_day', 'nunique'),
        first_watch_rel_day=('watch_rel_day', 'min'),
        last_watch_rel_day=('watch_rel_day', 'max'),
        avg_session_time=('watch_time', 'mean'),
        median_session_time=('watch_time', 'median'),
        min_session_time=('watch_time', 'min'),
        max_session_time=('watch_time', 'max'),
        std_session_time=('watch_time', 'std'),
    ).reset_index()

    basic['std_session_time'] = basic['std_session_time'].fillna(0)
    basic['active_span_days'] = (basic['last_watch_rel_day'] - basic['first_watch_rel_day'] + 1).clip(lower=0)
    basic['days_since_first_watch_from_reg'] = basic['first_watch_rel_day']
    basic['days_since_last_watch_to_obs_end'] = 20 - basic['last_watch_rel_day']
    basic['watch_days_ratio'] = basic['unique_days'] / 21

    basic['sessions_per_active_day'] = np.where(
        basic['unique_days'] > 0,
        basic['total_sessions'] / basic['unique_days'],
        0,
    )
    basic['contents_per_active_day'] = np.where(
        basic['unique_days'] > 0,
        basic['unique_contents'] / basic['unique_days'],
        0,
    )
    basic['avg_watch_time_per_content'] = np.where(
        basic['unique_contents'] > 0,
        basic['total_watch_time'] / basic['unique_contents'],
        0,
    )
    return basic

basic_usage = build_basic_usage_features(obs_view)
display(basic_usage.head())

,membership_row_id,total_watch_time,total_sessions,unique_contents,unique_days,first_watch_rel_day,last_watch_rel_day,avg_session_time,median_session_time,min_session_time,max_session_time,std_session_time,active_span_days,days_since_first_watch_from_reg,days_since_last_watch_to_obs_end,watch_days_ratio,sessions_per_active_day,contents_per_active_day,avg_watch_time_per_content
0,0,518,13,10,9,0,16,39.846154,13.0,1,200,57.936526,17,0,4,0.428571,1.444444,1.111111,51.800000
1,1,4,3,3,2,6,7,1.333333,1.0,1,2,0.577350,2,6,13,0.095238,1.500000,1.500000,1.333333
2,4,129,2,2,1,10,10,64.500000,64.5,1,128,89.802561,1,10,10,0.047619,2.000000,2.000000,64.500000
3,7,99,2,1,1,4,4,49.500000,49.5,15,84,48.790368,1,4,16,0.047619,2.000000,1.000000,99.000000
4,9,276,22,20,8,0,16,12.545455,2.0,1,106,24.849504,17,0,4,0.380952,2.750000,2.500000,13.800000


## 4-6. 일별 시청량과 몰아보기 피처

몰아보기 피처는 특정 하루에 시청시간이 몰리는지를 확인하기 위한 변수다. 총 시청시간 자체보다 시청이 특정 시점에 집중되는지가 이탈 행동과 더 관련될 수 있다.

In [11]:
def build_daily_usage_features(obs: pd.DataFrame) -> pd.DataFrame:
    daily = obs.groupby(['membership_row_id', 'watch_rel_day'], as_index=False).agg(
        daily_watch_time=('watch_time', 'sum'),
        daily_sessions=('watch_time', 'count'),
        daily_unique_contents=('MOVIE_NUM', 'nunique'),
    )

    daily_agg = daily.groupby('membership_row_id').agg(
        max_daily_watch_time=('daily_watch_time', 'max'),
        avg_daily_watch_time=('daily_watch_time', 'mean'),
        median_daily_watch_time=('daily_watch_time', 'median'),
        std_daily_watch_time=('daily_watch_time', 'std'),
        max_daily_sessions=('daily_sessions', 'max'),
    ).reset_index()

    daily_agg['std_daily_watch_time'] = daily_agg['std_daily_watch_time'].fillna(0)

    return daily, daily_agg

daily_usage_long, daily_usage = build_daily_usage_features(obs_view)
display(daily_usage.head())

,membership_row_id,max_daily_watch_time,avg_daily_watch_time,median_daily_watch_time,std_daily_watch_time,max_daily_sessions
0,0,201,57.555556,48.0,63.160730,2
1,1,3,2.000000,2.0,1.414214,2
2,4,129,129.000000,129.0,0.000000,2
3,7,99,99.000000,99.0,0.000000,2
4,9,106,34.500000,19.5,37.282704,10


## 4-7. 시청 간격과 공백 피처

OTT 이탈은 총량보다 습관성의 문제일 수 있다. 따라서 시청일 사이의 간격과 관측창 내 최대 비활성 구간을 계산한다.

In [12]:
def compute_gap_features(day_values: pd.Series) -> pd.Series:
    days = sorted(pd.Series(day_values).dropna().astype(int).unique())
    if len(days) == 0:
        return pd.Series({
            'avg_gap_between_watch_days': np.nan,
            'max_gap_between_watch_days': np.nan,
            'max_inactive_gap_days': 21,
            'start_inactive_days': 21,
            'end_inactive_days': 21,
        })

    start_inactive = days[0]
    end_inactive = 20 - days[-1]

    if len(days) >= 2:
        diffs = np.diff(days)
        avg_gap = float(np.mean(diffs))
        max_gap = float(np.max(diffs))
        internal_inactive = int(max(np.max(diffs) - 1, 0))
    else:
        avg_gap = np.nan
        max_gap = np.nan
        internal_inactive = 0

    max_inactive = max(int(start_inactive), int(end_inactive), int(internal_inactive))

    return pd.Series({
        'avg_gap_between_watch_days': avg_gap,
        'max_gap_between_watch_days': max_gap,
        'max_inactive_gap_days': max_inactive,
        'start_inactive_days': int(start_inactive),
        'end_inactive_days': int(end_inactive),
    })

watch_gap_features = (
    obs_view.groupby('membership_row_id')['watch_rel_day']
    .apply(compute_gap_features)
    .unstack()
    .reset_index()
)

watch_gap_features['avg_gap_between_watch_days'] = watch_gap_features['avg_gap_between_watch_days'].fillna(0)
watch_gap_features['max_gap_between_watch_days'] = watch_gap_features['max_gap_between_watch_days'].fillna(0)

display(watch_gap_features.head())

,membership_row_id,avg_gap_between_watch_days,max_gap_between_watch_days,max_inactive_gap_days,start_inactive_days,end_inactive_days
0,0,2.000000,5.0,4.0,0.0,4.0
1,1,1.000000,1.0,13.0,6.0,13.0
2,4,0.000000,0.0,10.0,10.0,10.0
3,7,0.000000,0.0,16.0,4.0,16.0
4,9,2.285714,5.0,4.0,0.0,4.0


## 4-8. 주차별 시청 피처

관측창 day 0~20을 1주차, 2주차, 3주차로 나누어 시청시간, 세션 수, 고유 콘텐츠 수, 활성일 수를 계산한다.

In [13]:
def pivot_weekly(obs: pd.DataFrame, value_col: str, aggfunc, prefix: str) -> pd.DataFrame:
    table = (
        obs.pivot_table(
            index='membership_row_id',
            columns='obs_week',
            values=value_col,
            aggfunc=aggfunc,
            fill_value=0,
        )
        .reset_index()
    )
    rename = {w: f'week{int(w)}_{prefix}' for w in [1, 2, 3] if w in table.columns}
    table = table.rename(columns=rename)
    for w in [1, 2, 3]:
        col = f'week{w}_{prefix}'
        if col not in table.columns:
            table[col] = 0
    return table[['membership_row_id', f'week1_{prefix}', f'week2_{prefix}', f'week3_{prefix}']]

weekly_watch = pivot_weekly(obs_view, 'watch_time', 'sum', 'watch_time')
weekly_sessions = pivot_weekly(obs_view, 'watch_time', 'count', 'sessions')
weekly_contents = pivot_weekly(obs_view, 'MOVIE_NUM', pd.Series.nunique, 'unique_contents')
weekly_days = pivot_weekly(obs_view, 'watch_rel_day', pd.Series.nunique, 'active_days')

weekly_features = (
    weekly_watch
    .merge(weekly_sessions, on='membership_row_id', how='outer')
    .merge(weekly_contents, on='membership_row_id', how='outer')
    .merge(weekly_days, on='membership_row_id', how='outer')
    .fillna(0)
)

display(weekly_features.head())

obs_week,membership_row_id,week1_watch_time,week2_watch_time,week3_watch_time,week1_sessions,week2_sessions,week3_sessions,week1_unique_contents,week2_unique_contents,week3_unique_contents,week1_active_days,week2_active_days,week3_active_days
0,0,191,313,14,3,8,2,2,8,2,3,5,1
1,1,3,1,0,2,1,0,2,1,0,1,1,0
2,4,0,129,0,0,2,0,0,2,0,0,1,0
3,7,99,0,0,2,0,0,1,0,0,1,0,0
4,9,26,80,170,3,8,11,3,8,9,2,4,2


## 4-9. 초반 루틴화, 후반 몰아보기, 변화량 피처

이 단계에서 프로젝트의 핵심 행동 가설을 수치화한다. 이전 탐색에서 단순 시청량보다 1주차 비중, 3주차 증가량, 후반 몰아보기 패턴이 더 의미 있는 후보로 나타났다.

In [14]:
def add_weekly_pattern_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    for col in ['week1_watch_time', 'week2_watch_time', 'week3_watch_time']:
        if col not in out.columns:
            out[col] = 0

    total_week_watch = out[['week1_watch_time', 'week2_watch_time', 'week3_watch_time']].sum(axis=1)

    out['week1_ratio'] = np.where(total_week_watch > 0, out['week1_watch_time'] / total_week_watch, 0)
    out['week2_ratio'] = np.where(total_week_watch > 0, out['week2_watch_time'] / total_week_watch, 0)
    out['week3_ratio'] = np.where(total_week_watch > 0, out['week3_watch_time'] / total_week_watch, 0)

    out['front_loaded_ratio'] = out['week1_ratio']
    out['late_ratio'] = out['week3_ratio']

    out['w2_minus_w1_watch_time'] = out['week2_watch_time'] - out['week1_watch_time']
    out['w3_minus_w2_watch_time'] = out['week3_watch_time'] - out['week2_watch_time']
    out['w3_minus_w1_watch_time'] = out['week3_watch_time'] - out['week1_watch_time']

    out['w3_to_w1_ratio'] = np.where(
        out['week1_watch_time'] > 0,
        out['week3_watch_time'] / out['week1_watch_time'],
        np.where(out['week3_watch_time'] > 0, np.inf, 0),
    )
    out['w3_to_w1_ratio_capped'] = out['w3_to_w1_ratio'].replace(np.inf, 999).clip(upper=999)

    # 1주차 중심점 day 3, 3주차 중심점 day 17 사이의 변화량을 간단한 slope로 둔다.
    out['daily_watch_slope'] = out['w3_minus_w1_watch_time'] / 14

    out['week_count_with_watch'] = (
        (out['week1_watch_time'] > 0).astype(int)
        + (out['week2_watch_time'] > 0).astype(int)
        + (out['week3_watch_time'] > 0).astype(int)
    )

    out['front_loaded_flag'] = (out['week1_ratio'] >= 0.5).astype(int)
    out['late_binge_flag'] = (out['week3_ratio'] >= 0.5).astype(int)
    out['only_week1_flag'] = ((out['week1_watch_time'] > 0) & (out['week2_watch_time'] == 0) & (out['week3_watch_time'] == 0)).astype(int)
    out['only_week3_flag'] = ((out['week1_watch_time'] == 0) & (out['week2_watch_time'] == 0) & (out['week3_watch_time'] > 0)).astype(int)
    out['no_week1_flag'] = (out['week1_watch_time'] == 0).astype(int)
    out['no_week3_flag'] = (out['week3_watch_time'] == 0).astype(int)
    out['steady_3week_watch_flag'] = (out['week_count_with_watch'] == 3).astype(int)

    out['usage_trend_label'] = np.select(
        [
            out['only_week1_flag'] == 1,
            out['only_week3_flag'] == 1,
            out['w3_minus_w1_watch_time'] > 30,
            out['w3_minus_w1_watch_time'] < -30,
            out['steady_3week_watch_flag'] == 1,
        ],
        [
            'only_week1',
            'only_week3',
            'late_increasing',
            'front_decreasing',
            'steady_all_weeks',
        ],
        default='mixed_or_low_activity',
    )

    return out

## 4-10. 전체 usage feature table 조립

In [15]:
usage_features = (
    basic_usage
    .merge(daily_usage, on='membership_row_id', how='left')
    .merge(watch_gap_features, on='membership_row_id', how='left')
    .merge(weekly_features, on='membership_row_id', how='left')
)

usage_features = add_weekly_pattern_features(usage_features)

usage_features['max_day_share'] = np.where(
    usage_features['total_watch_time'] > 0,
    usage_features['max_daily_watch_time'] / usage_features['total_watch_time'],
    0,
)
usage_features['one_day_binge_flag'] = (usage_features['max_day_share'] >= 0.8).astype(int)
usage_features['has_usage_feature'] = 1

# Ensure stable column order.
front_cols = ['membership_row_id', 'has_usage_feature']
usage_features = usage_features[front_cols + [c for c in usage_features.columns if c not in front_cols]]

usage_features = usage_features.replace([np.inf, -np.inf], np.nan)

display(usage_features.head())
print('usage_features shape:', usage_features.shape)

,membership_row_id,has_usage_feature,total_watch_time,total_sessions,unique_contents,unique_days,first_watch_rel_day,last_watch_rel_day,avg_session_time,median_session_time,...,front_loaded_flag,late_binge_flag,only_week1_flag,only_week3_flag,no_week1_flag,no_week3_flag,steady_3week_watch_flag,usage_trend_label,max_day_share,one_day_binge_flag
0,0,1,518,13,10,9,0,16,39.846154,13.0,...,0,0,0,0,0,0,1,front_decreasing,0.388031,0
1,1,1,4,3,3,2,6,7,1.333333,1.0,...,1,0,0,0,0,1,0,mixed_or_low_activity,0.750000,0
2,4,1,129,2,2,1,10,10,64.500000,64.5,...,0,0,0,0,1,1,0,mixed_or_low_activity,1.000000,1
3,7,1,99,2,1,1,4,4,49.500000,49.5,...,1,0,1,0,0,1,0,only_week1,1.000000,1
4,9,1,276,22,20,8,0,16,12.545455,2.0,...,0,1,0,0,0,0,1,late_increasing,0.384058,0


usage_features shape: (12302, 64)


## 4-11. 시청이력 없는 고객 포함 모델링 테이블 생성

시청이력 없는 고객은 삭제하지 않는다. 04번에서 생성한 피처를 membership table에 left join하고, 시청이력 없는 고객의 행동 피처는 0 또는 별도 플래그로 채운다.

In [16]:
modeling_usage = membership.merge(usage_features, on='membership_row_id', how='left')

usage_feature_cols = [c for c in usage_features.columns if c not in ['membership_row_id']]

# 시청이력 없는 고객의 수치형 행동 피처는 0으로 채운다.
for col in usage_feature_cols:
    if col == 'usage_trend_label':
        modeling_usage[col] = modeling_usage[col].fillna('no_watch')
    else:
        modeling_usage[col] = modeling_usage[col].fillna(0)

modeling_usage['has_usage_feature'] = modeling_usage['has_usage_feature'].astype(int)
modeling_usage['no_usage_feature_flag'] = (modeling_usage['has_usage_feature'] == 0).astype(int)

# 02번의 has_watch_obs와 04번의 has_usage_feature가 같은지 검산할 수 있도록 유지한다.
watch_flag_check = pd.crosstab(
    modeling_usage['has_watch_obs'],
    modeling_usage['has_usage_feature'],
    rownames=['has_watch_obs_from_02'],
    colnames=['has_usage_feature_from_04'],
)

display(watch_flag_check)
watch_flag_check.to_csv(TABLES_DIR / '04_usage_feature_watch_flag_check.csv', encoding='utf-8-sig')

print('modeling_usage shape:', modeling_usage.shape)

has_usage_feature_from_04,0,1
has_watch_obs_from_02,,
0,2620,0
1,0,12302


modeling_usage shape: (14922, 92)


## 4-12. 가설형 행동 세그먼트 후보 생성

이 노트북에서는 콘텐츠 메타데이터 없이 만들 수 있는 행동 세그먼트 후보만 생성한다. 콘텐츠 기반 세그먼트는 05번 이후에 만든다.

In [17]:
modeling_usage['stable_2screen_active'] = (
    (modeling_usage['max_screen'] == 2)
    & (modeling_usage['unique_days'] >= 3)
    & (modeling_usage['week1_ratio'] > 0)
).astype(int)

modeling_usage['discount_sensitive_risk'] = (
    (modeling_usage['is_100won'] == 1)
    & (modeling_usage['max_screen'] == 4)
    & (modeling_usage['week3_ratio'] >= 0.5)
).astype(int)

modeling_usage['promo4_late_increasing_risk'] = (
    (modeling_usage['is_100won'] == 1)
    & (modeling_usage['max_screen'] == 4)
    & (modeling_usage['w3_minus_w1_watch_time'] > 0)
).astype(int)

modeling_usage['promo2_early_routine_candidate'] = (
    (modeling_usage['is_100won'] == 1)
    & (modeling_usage['max_screen'] == 2)
    & (modeling_usage['week1_ratio'] >= 0.3)
    & (modeling_usage['unique_days'] >= 3)
).astype(int)

segment_flags = [
    'stable_2screen_active',
    'discount_sensitive_risk',
    'promo4_late_increasing_risk',
    'promo2_early_routine_candidate',
]

segment_check = []
for col in segment_flags:
    tmp = modeling_usage.groupby(col)['is_repurchase'].agg(n='count', repurchase_rate='mean').reset_index()
    tmp.insert(0, 'segment_flag', col)
    tmp = tmp.rename(columns={col: 'flag_value'})
    segment_check.append(tmp)
segment_check = pd.concat(segment_check, ignore_index=True)

display(segment_check)
segment_check.to_csv(TABLES_DIR / '04_usage_feature_behavior_segment_flag_rates.csv', index=False, encoding='utf-8-sig')

,segment_flag,flag_value,n,repurchase_rate
0,stable_2screen_active,0,13595,0.667893
1,stable_2screen_active,1,1327,0.750565
2,discount_sensitive_risk,0,14407,0.682793
3,discount_sensitive_risk,1,515,0.464078
4,promo4_late_increasing_risk,0,14161,0.686745
5,promo4_late_increasing_risk,1,761,0.461235
6,promo2_early_routine_candidate,0,14490,0.672671
7,promo2_early_routine_candidate,1,432,0.761574


## 4-13. 주요 피처 검산표 생성

06번 유의성 검정에 들어가기 전, 주요 행동 피처들의 분포와 결측 상태를 확인한다.

In [18]:
usage_numeric_cols = [
    'total_watch_time', 'total_sessions', 'unique_contents', 'unique_days',
    'avg_session_time', 'max_session_time', 'active_span_days', 'watch_days_ratio',
    'sessions_per_active_day', 'contents_per_active_day',
    'max_daily_watch_time', 'max_day_share', 'days_since_last_watch_to_obs_end',
    'avg_gap_between_watch_days', 'max_inactive_gap_days',
    'week1_watch_time', 'week2_watch_time', 'week3_watch_time',
    'week1_ratio', 'week2_ratio', 'week3_ratio',
    'w3_minus_w1_watch_time', 'daily_watch_slope',
]
usage_numeric_cols = [c for c in usage_numeric_cols if c in modeling_usage.columns]

feature_summary = modeling_usage[usage_numeric_cols].describe().T.reset_index().rename(columns={'index': 'feature'})
feature_summary['missing_count'] = modeling_usage[usage_numeric_cols].isna().sum().values
feature_summary['zero_count'] = (modeling_usage[usage_numeric_cols] == 0).sum().values
feature_summary['zero_rate'] = feature_summary['zero_count'] / len(modeling_usage)

feature_summary.to_csv(TABLES_DIR / '04_usage_feature_numeric_summary.csv', index=False, encoding='utf-8-sig')
display(feature_summary.head(30))

,feature,count,mean,std,min,25%,50%,75%,max,missing_count,zero_count,zero_rate
0,total_watch_time,14922.0,254.680941,313.587752,0.0,11.000000,155.000000,370.000000,4419.000000,0,2620,0.175580
1,total_sessions,14922.0,5.700710,5.871141,0.0,1.000000,4.000000,8.000000,56.000000,0,2620,0.175580
2,unique_contents,14922.0,4.091945,4.210402,0.0,1.000000,3.000000,6.000000,47.000000,0,2620,0.175580
3,unique_days,14922.0,3.121632,2.678445,0.0,1.000000,3.000000,5.000000,17.000000,0,2620,0.175580
4,avg_session_time,14922.0,37.237338,32.854482,0.0,4.000000,33.763889,58.250000,200.000000,0,2620,0.175580
5,max_session_time,14922.0,79.338762,59.687906,0.0,7.000000,93.000000,122.000000,200.000000,0,2620,0.175580
6,active_span_days,14922.0,8.571840,7.274872,0.0,1.000000,8.000000,15.000000,21.000000,0,2620,0.175580
7,watch_days_ratio,14922.0,0.148649,0.127545,0.0,0.047619,0.142857,0.238095,0.809524,0,2620,0.175580
8,sessions_per_active_day,14922.0,1.496231,1.149460,0.0,1.000000,1.400000,2.000000,18.000000,0,2620,0.175580
9,contents_per_active_day,14922.0,1.131992,0.937069,0.0,0.666667,1.000000,1.500000,15.000000,0,2620,0.175580


In [19]:
usage_group_summary = modeling_usage.groupby('is_100won').agg(
    n=('membership_row_id', 'count'),
    repurchase_rate=('is_repurchase', 'mean'),
    avg_total_watch_time=('total_watch_time', 'mean'),
    avg_unique_days=('unique_days', 'mean'),
    avg_week1_ratio=('week1_ratio', 'mean'),
    avg_week3_ratio=('week3_ratio', 'mean'),
    avg_w3_minus_w1=('w3_minus_w1_watch_time', 'mean'),
    no_watch_rate=('no_watch_obs_flag', 'mean'),
).reset_index()

usage_screen_summary = modeling_usage.groupby(['is_100won', 'max_screen'], dropna=False).agg(
    n=('membership_row_id', 'count'),
    repurchase_rate=('is_repurchase', 'mean'),
    avg_total_watch_time=('total_watch_time', 'mean'),
    avg_unique_days=('unique_days', 'mean'),
    avg_week1_ratio=('week1_ratio', 'mean'),
    avg_week3_ratio=('week3_ratio', 'mean'),
    avg_w3_minus_w1=('w3_minus_w1_watch_time', 'mean'),
    no_watch_rate=('no_watch_obs_flag', 'mean'),
).reset_index()

usage_group_summary.to_csv(TABLES_DIR / '04_usage_feature_summary_by_100won.csv', index=False, encoding='utf-8-sig')
usage_screen_summary.to_csv(TABLES_DIR / '04_usage_feature_summary_by_100won_maxscreen.csv', index=False, encoding='utf-8-sig')

display(usage_group_summary)
display(usage_screen_summary)

,is_100won,n,repurchase_rate,avg_total_watch_time,avg_unique_days,avg_week1_ratio,avg_week3_ratio,avg_w3_minus_w1,no_watch_rate
0,0,5939,0.741034,252.704664,3.108436,0.285992,0.272458,-2.170567,0.172925
1,1,8983,0.631749,255.987532,3.130357,0.281728,0.280724,-2.479127,0.177335


,is_100won,max_screen,n,repurchase_rate,avg_total_watch_time,avg_unique_days,avg_week1_ratio,avg_week3_ratio,avg_w3_minus_w1,no_watch_rate
0,0,1,3794,0.719294,254.135741,3.108593,0.288030,0.272738,-2.990511,0.170532
1,0,2,1594,0.784191,254.609159,3.106023,0.283943,0.270066,-3.530740,0.186324
2,0,4,551,0.765880,237.341198,3.114338,0.277889,0.277455,7.410163,0.150635
3,1,1,5413,0.658045,256.572511,3.124515,0.282691,0.280987,-1.803621,0.176058
4,1,2,1528,0.740183,270.200916,3.232984,0.287723,0.278412,-6.836387,0.176047
5,1,4,2042,0.480901,243.801175,3.069050,0.274688,0.281755,-1.009305,0.181685


## 4-14. 산출물 저장

In [20]:
PATH_USAGE_FEATURES = OUTPUT_DATA_DIR / "usage_features.csv"
PATH_MODELING_USAGE = OUTPUT_DATA_DIR / "modeling_feature_table_usage.csv"
PATH_USAGE_SUMMARY_JSON = OUTPUT_DATA_DIR / "usage_feature_summary.json"

usage_features.to_csv(PATH_USAGE_FEATURES, index=False, encoding="utf-8-sig")
modeling_usage.to_csv(PATH_MODELING_USAGE, index=False, encoding="utf-8-sig")

summary = {
    "inputs": {k: str(v) for k, v in INPUT_FILES.items()},
    "outputs": {
        "usage_features": str(PATH_USAGE_FEATURES),
        "modeling_feature_table_usage": str(PATH_MODELING_USAGE),
        "usage_feature_summary": str(PATH_USAGE_SUMMARY_JSON),
    },
    "rows": {
        "membership_preprocessed": int(len(membership)),
        "obs_view": int(len(obs_view)),
        "usage_features": int(len(usage_features)),
        "modeling_usage": int(len(modeling_usage)),
    },
    "feature_counts": {
        "usage_feature_columns": int(len(usage_features.columns)),
        "modeling_usage_columns": int(len(modeling_usage.columns)),
    },
    "watch_presence": {
        "has_usage_feature_rows": int(modeling_usage["has_usage_feature"].sum()),
        "no_usage_feature_rows": int(modeling_usage["no_usage_feature_flag"].sum()),
    },
    "key_rates": {
        "overall_repurchase_rate": float(modeling_usage["is_repurchase"].mean()),
        "100won_repurchase_rate": float(modeling_usage.loc[modeling_usage["is_100won"] == 1, "is_repurchase"].mean()),
        "non_100won_repurchase_rate": float(modeling_usage.loc[modeling_usage["is_100won"] == 0, "is_repurchase"].mean()),
    },
}

with open(PATH_USAGE_SUMMARY_JSON, "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

expected_report_files = {
    "04_usage_feature_input_file_summary.csv",
    "04_usage_feature_observation_input_check.csv",
    "04_usage_feature_watch_flag_check.csv",
    "04_usage_feature_behavior_segment_flag_rates.csv",
    "04_usage_feature_numeric_summary.csv",
    "04_usage_feature_summary_by_100won.csv",
    "04_usage_feature_summary_by_100won_maxscreen.csv",
    "04_usage_feature_final_checks.csv",
}

final_checks_path = OUTPUT_TABLE_DIR / "04_usage_feature_final_checks.csv"
actual_04_report_files = {p.name for p in OUTPUT_TABLE_DIR.glob("04_*.csv")} | {final_checks_path.name}

has_watch_obs_int = modeling_usage["has_watch_obs"].astype(int)
has_usage_feature_int = modeling_usage["has_usage_feature"].astype(int)

final_checks = pd.DataFrame([
    {"check": "project_root_is_repo_root", "value": str(PROJECT_ROOT), "pass": (PROJECT_ROOT / ".git").exists()},
    {"check": "data_root_is_repo_data", "value": str(DATA_ROOT), "pass": DATA_ROOT == PROJECT_ROOT / "_data" and DATA_ROOT.exists()},
    {"check": "reports_dir_is_park_reports", "value": str(REPORTS_DIR), "pass": REPORTS_DIR == WORK_ROOT / "reports"},
    {"check": "input_data_dir_is_02_reports_data", "value": str(INPUT_DATA_DIR), "pass": INPUT_DATA_DIR == REPORTS_DIR / "data" / "02_preprocessing_policy"},
    {"check": "output_data_dir_under_reports_data_04", "value": str(OUTPUT_DATA_DIR), "pass": OUTPUT_DATA_DIR == REPORTS_DIR / "data" / NOTEBOOK_ID},
    {"check": "output_table_dir_under_reports_tables_04", "value": str(OUTPUT_TABLE_DIR), "pass": OUTPUT_TABLE_DIR == REPORTS_DIR / "tables" / NOTEBOOK_ID},
    {"check": "only_expected_04_report_csvs", "value": sorted(actual_04_report_files), "pass": actual_04_report_files == expected_report_files},
    {"check": "membership_rows_preserved", "value": len(modeling_usage), "pass": len(modeling_usage) == len(membership)},
    {"check": "membership_row_id_unique", "value": bool(modeling_usage["membership_row_id"].is_unique), "pass": bool(modeling_usage["membership_row_id"].is_unique)},
    {"check": "usage_features_rows_match_has_watch_obs", "value": len(usage_features), "pass": len(usage_features) == int(membership["has_watch_obs"].sum())},
    {"check": "has_usage_feature_matches_has_watch_obs", "value": str(pd.crosstab(has_watch_obs_int, has_usage_feature_int).to_dict()), "pass": bool((has_watch_obs_int == has_usage_feature_int).all())},
    {"check": "obs_watch_rel_day_min", "value": int(obs_view["watch_rel_day"].min()) if len(obs_view) else None, "pass": len(obs_view) > 0 and obs_view["watch_rel_day"].min() >= 0},
    {"check": "obs_watch_rel_day_max", "value": int(obs_view["watch_rel_day"].max()) if len(obs_view) else None, "pass": len(obs_view) > 0 and obs_view["watch_rel_day"].max() <= 20},
    {"check": "usage_features_output_exists", "value": str(PATH_USAGE_FEATURES), "pass": PATH_USAGE_FEATURES.exists()},
    {"check": "modeling_usage_output_exists", "value": str(PATH_MODELING_USAGE), "pass": PATH_MODELING_USAGE.exists()},
    {"check": "summary_json_output_exists", "value": str(PATH_USAGE_SUMMARY_JSON), "pass": PATH_USAGE_SUMMARY_JSON.exists()},
])

final_checks.to_csv(final_checks_path, index=False, encoding="utf-8-sig")
display(final_checks)

if not final_checks["pass"].all():
    failed = final_checks.loc[~final_checks["pass"]]
    raise AssertionError(f"04번 최종 검산 실패:\n{failed}")

print("Saved data outputs to:", OUTPUT_DATA_DIR)
print("Saved report tables to:", OUTPUT_TABLE_DIR)
print("04_usage_feature_engineering passed final checks.")


,check,value,pass
0,project_root_is_repo_root,c:\Code\ott-churn-prediction,True
1,data_root_is_repo_data,c:\Code\ott-churn-prediction\_data,True
2,reports_dir_is_park_reports,c:\Code\ott-churn-prediction\park.ingyeom\reports,True
3,input_data_dir_is_02_reports_data,c:\Code\ott-churn-prediction\park.ingyeom\repo...,True
4,output_data_dir_under_reports_data_04,c:\Code\ott-churn-prediction\park.ingyeom\repo...,True
5,output_table_dir_under_reports_tables_04,c:\Code\ott-churn-prediction\park.ingyeom\repo...,True
6,only_expected_04_report_csvs,[04_usage_feature_behavior_segment_flag_rates....,True
7,membership_rows_preserved,14922,True
8,membership_row_id_unique,True,True
9,usage_features_rows_match_has_watch_obs,12302,True


Saved data outputs to: c:\Code\ott-churn-prediction\park.ingyeom\reports\data\04_usage_feature_engineering
Saved report tables to: c:\Code\ott-churn-prediction\park.ingyeom\reports\tables\04_usage_feature_engineering
04_usage_feature_engineering passed final checks.


## 4-15. 04번 노트북 결론

04번 노트북의 최종 산출물은 `modeling_feature_table_usage.csv`이다.  
이 파일은 `membership_row_id` 기준 구독 이벤트 단위의 멤버십 정보와 1~3주차 시청 행동 피처를 결합한 테이블이다.

저장 위치:

```text
park.ingyeom/reports/data/04_usage_feature_engineering/modeling_feature_table_usage.csv
park.ingyeom/reports/data/04_usage_feature_engineering/usage_features.csv
park.ingyeom/reports/data/04_usage_feature_engineering/usage_feature_summary.json
park.ingyeom/reports/tables/04_usage_feature_engineering/
```

다음 05번 노트북은 03번의 영화 메타데이터 산출물과 04번의 usage feature 산출물을 함께 사용해 콘텐츠 성향 피처를 생성한다.
